# PySpark full outer join



# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("FullOuterJOin") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/05/10 22:51:10 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/10 22:51:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7ac080bb-48a4-4cf0-a939-406c01ceed01;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 361ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0

# Create 2 dataframe of test data

In [2]:
employees = spark.createDataFrame(
    [(1, "Alice"), (2, "Bob"), (3, "Carol")],
    ["id", "name"]
)

departments = spark.createDataFrame(
    [(2, "Engineering"), (3, "Marketing"), (4, "Sales")],
    ["id", "dept"]
)

# Full outer join on the id between the 2 dataframes


In [5]:
result = employees.join(departments, on=employees.id == departments.id, how="full_outer")
result.show()

[Stage 6:>                                                        (0 + 16) / 16]

+----+-----+----+-----------+
|  id| name|  id|       dept|
+----+-----+----+-----------+
|   1|Alice|NULL|       NULL|
|   2|  Bob|   2|Engineering|
|   3|Carol|   3|  Marketing|
|NULL| NULL|   4|      Sales|
+----+-----+----+-----------+



# Fill null with something more appropriate

In [11]:
result_filled = result.fillna("N/A").fillna({"id": 0})
result_filled.show()


AnalysisException: [AMBIGUOUS_REFERENCE] Reference `id` is ambiguous, could be: [`id`, `id`].